<a href="https://colab.research.google.com/github/trianitriani/EasyDrugServer/blob/main/YouTrend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#[1] Setup ambiente
## [1.1] Libraries

In [13]:
import pandas as pd
import numpy as np
import os
import re
import ast
from google.colab import drive

from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
!pip3 install pyclustering
!pip3 install torch
from pyclustering.cluster.kmedoids import kmedoids
#Risolvere problema dipendenze
#!pip install scikit-learn-extra
#from sklearn_extra.cluster import CLARA
from pyclustering.cluster.clarans import clarans
from sklearn.cluster import AgglomerativeClustering #AGNES
#DIANA : implementazione su github a https://github.com/div338/Divisive-Clustering-Analysis-Program-DIANA-
from sklearn.cluster import Birch
#Chameleon : implementazione su github a https://github.com/Moonpuck/chameleon_cluster
from sklearn.cluster import DBSCAN
from sklearn.cluster import OPTICS
#DENCLUE : implementazione su https://www.kaggle.com/code/hassanashfaq2001/denclue-clustering-algorithm
from pyclustering.cluster.clique import clique
#STING : nessuna implementazione trovata
from sklearn.cluster import HDBSCAN


from typing import Union
from sklearn.neighbors import BallTree
import time
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.metrics import adjusted_rand_score as ARI
from sklearn.metrics.cluster import contingency_matrix as CM
from sklearn.metrics import homogeneity_completeness_v_measure as HCV
from sentence_transformers import SentenceTransformer, util
import torch


##[1.2] Space setting


In [5]:
drive.mount('/content/drive')
path_prefix=os.path.join('/content/drive','MyDrive','Data mining')

Mounted at /content/drive


## [1.4] Utilities

In [6]:
def profile(f):
    def f_timer(*args, **kwargs):
        start = time.time()
        result = f(*args, **kwargs)
        end = time.time()
        ms = (end - start) * 1000
        print(f"{f.__name__} ({ms:.3f} ms)")
        return result
    return f_timer


def csv_compress_IO(filename,IOtype):
  """
  Function used to open or save a csv file in a compress way (allows to push it to github repo)
  """
  if IOtype=="open":
    print("File opened successfully")
  elif IOtype=="save":
    print("File saved successfully")
  else:
    print("Wrong IOtype")


#[2] Pre-Processing
Here will be putted functions for:
- data pre-processing
- data integration

In [24]:
def linkParserList(original):
    '''
    Utility function used to extract pure categories from wikipedia link that embed those names
    :return: string like "['category1','category2',...,'categoryN']"
    '''
    topics = ""
    lastword = ""
    inizio = 0
    if isinstance(original, float):
        return "[]"

    for elem in original:
        if elem == '[' or elem == ']' or elem == "'" or elem == "," or elem == " ":
            if elem == "'":
                if inizio == 0:
                    inizio = 1
                else:
                    inizio = 0
                    topics += lastword
            topics += elem
            continue
        if elem == ']':
            topics += elem
            break
        if elem == "/":
            lastword = ""
            continue
        lastword += elem
    return topics

def videoDocumentExtraction(csvName,sanitizationlvl=0):
    '''
    Take the dataset containing information on youtube videos,
    generate documents for each videos that contain the knowledge
    and collect them in a dataframe with compact information

    :param sanitizationlvl: specify the level of sanitization in the knoledge extracted from videos (e.g remove tags, link...). Default doesn't sanitize.
    :return: dataframe [channelId,videoId,content]
    '''
    description_clear = ""
    description_tags = []
    video_path=os.path.join(path_prefix,"Results",csvName)
    videosDf = pd.read_csv(video_path)
    documentList=[]
    taggerList=[]
    topicsList=[]
    for index, row in videosDf.iterrows():
        if pd.isna(row["description"]):
            description_clear=""
        else:
            # remove the links from the description
            description_clear = re.sub(r"http\S+", "", row["description"])
            description_clear = description_clear.lower()
            # finds the tag in the description
            description_tags = re.findall(r"#\w+", description_clear)
            description_tags = [t[1:].lower() for t in description_tags]
            # remove all the tags in the description and remove double space
            description_clear = re.sub(r"#\w+", "", description_clear)
            description_clear = re.sub(r"\s+", " ", description_clear).strip()

        # this will be the content on which sbert works
        document = row["title"] + '\n' + description_clear
        documentList.append(document)

        # now we have to create a list of tagger
        topics = linkParserList(row["topicCategories"])
        topics = [t.lower() for t in ast.literal_eval(topics)]
        tags = [t.lower() for t in ast.literal_eval(row["tags"])]
        topicsList.append(topics)
        """for topic in topics:
            if topic not in tags:
                tags.append(topic)"""

        if not description_clear:
            for description_tag in description_tags:
                if description_tag not in tags:
                    tags.append(description_tag)
        taggerList.append(tags)

    documentsDf = videosDf.copy()
    documentsDf = documentsDf[["channelId", "videoId"]]
    documentsDf["content"]=documentList
    documentsDf["tags"]=taggerList
    documentsDf["topics"]=topicsList
    documentsDf.columns=["channelId","videoId","content", "tags", "topics"]
    return documentsDf

videos=videoDocumentExtraction("videos_youtuber_us.csv")
videos.to_csv(os.path.join(path_prefix,"Results","videos_documents.csv"),index=False)
print(videos.head())

                  channelId      videoId  \
0  UCf1Q757aKSmywQDQhiTtGFA  5IewOvaQtTg   
1  UCf1Q757aKSmywQDQhiTtGFA  E9d-pSU0HLs   
2  UCf1Q757aKSmywQDQhiTtGFA  pcnzj2RNbo0   
3  UCf1Q757aKSmywQDQhiTtGFA  TM6CInJx-XU   
4  UCf1Q757aKSmywQDQhiTtGFA  Ox90jpnHhIk   

                                             content  \
0  Yungeen Ace - Im Sorry (Official Video)\nliste...   
1  Yungeen Ace -  Under The Street Lights (Offici...   
2  Yungeen Ace - Sacrifice Erverybody (Official A...   
3  Yungeen Ace - Deep Thoughts (Official Audio)\n...   
4  Yungeen Ace - Oh Way Oh No (Official Audio)\nl...   

                                                tags  \
0  [yungeen ace, jacksonville florida, florida, j...   
1  [yungeen ace, jacksonville florida, florida, j...   
2  [yungeen ace, jacksonville florida, florida, j...   
3  [yungeen ace, jacksonville florida, florida, j...   
4  [yungeen ace, jacksonville florida, florida, j...   

                                              topics  
0     

Innanzitutto controllo la quantità di video senza categorie, le quantifico e le rimuovo perché non possono aiutarmi per effettuare la fase successiva del fine tuning.

In [25]:
import pandas as pd
import numpy as np

# Supponiamo che 'doc' sia il DataFrame ottenuto dal tuo codice
total_videos = len(videos)

# Controlliamo quali righe hanno topics vuoti o NaN
empty_topics_mask = videos["topics"].isna() | videos["topics"].apply(lambda x: isinstance(x, list) and len(x) == 0)

# Conteggio
num_empty = empty_topics_mask.sum()

# Percentuale
perc_empty = (num_empty / total_videos) * 100

print(f"Numero di video senza topic: {num_empty} ({perc_empty:.2f}% del totale)")

Numero di video senza topic: 2370 (3.11% del totale)


In [26]:
# Crea la maschera per identificare i video con topics vuoti
empty_topics_mask = videos["topics"].isna() | videos["topics"].apply(lambda x: isinstance(x, list) and len(x) == 0)

# Rimuovi quelle righe
videos = videos[~empty_topics_mask].reset_index(drop=True)

Adesso voglio provare a effettuare un fine tuning di sbert usando una parte del dataset che ho, usando come etichetta di riferimento le varie categorie che sono state fornite da youtube.



Queste funzioni vengono utilizzate per controllare la similarità tra due video youtube usando come ground truth le liste di categoie tra due video diversi

In [16]:
model_for_similarity = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_similarity(cat_list1, cat_list2):
    # Se una delle due liste è vuota, la similarità è 0
    if not cat_list1 or not cat_list2:
        return 0.0

    # Encode ogni lista in un singolo vettore medio
    emb1 = model_for_similarity.encode(cat_list1, convert_to_tensor=True)
    emb2 = model_for_similarity.encode(cat_list2, convert_to_tensor=True)

    # Media dei vettori dei topic
    emb1_mean = torch.mean(emb1, dim=0)
    emb2_mean = torch.mean(emb2, dim=0)

    # Similarità coseno
    return util.cos_sim(emb1_mean, emb2_mean).item()

def semantic_similarity_pairwise(cat_list1, cat_list2):
    if not cat_list1 or not cat_list2:
        return 0.0

    emb1 = model_for_similarity.encode(cat_list1, convert_to_tensor=True)
    emb2 = model_for_similarity.encode(cat_list2, convert_to_tensor=True)

    # Matrice delle cosine pairwise
    cos_matrix = util.cos_sim(emb1, emb2)

    # Media di tutte le similarità (approccio simmetrico)
    return torch.mean(cos_matrix).item()

Vediamo qui un esempio di come funzionano date queste liste di categorie come esempio, il modello di sbert fornisce una valutazione rispetto alla semantica.

In [19]:
video1_topics = ["music", "pop music", "entertainment"]
video2_topics = ["rock music", "concert", "live performance"]
video3_topics = ["gaming", "action game", "video game culture"]
video4_topics = ["music", "pop music"]

print("🎵 vs 🎵", semantic_similarity(video1_topics, video2_topics))
print("🎵 vs 🎮", semantic_similarity(video1_topics, video3_topics))
print("🎵 vs 🎵", semantic_similarity(video1_topics, video4_topics))
print("\n")
print("🎵 vs 🎵", semantic_similarity_pairwise(video1_topics, video2_topics))
print("🎵 vs 🎮", semantic_similarity_pairwise(video1_topics, video3_topics))
print("🎵 vs 🎵", semantic_similarity_pairwise(video1_topics, video4_topics))


🎵 vs 🎵 0.747611403465271
🎵 vs 🎮 0.5509433746337891
🎵 vs 🎵 0.9391571283340454


🎵 vs 🎵 0.4918629825115204
🎵 vs 🎮 0.3927229940891266
🎵 vs 🎵 0.7396847009658813


In [41]:
from sklearn.model_selection import train_test_split

# spostare i video di youtuber con un solo video nel test set altrimenti non
# può essere stratificato
counts = videos["channelId"].value_counts()

single_video_channels = counts[counts == 1].index
multi_video_channels  = counts[counts > 1].index

single_video_df = videos[videos["channelId"].isin(single_video_channels)]
multi_video_df  = videos[videos["channelId"].isin(multi_video_channels)]

print(f"The total youtubers in the collection are: {len(counts)}")
print(f"The youtuber with only one video: {len(single_video_df)}")

# mantieni distribuzione per canale (e magari anche sui topic)
# suppongo che di solito un canale youtube produca video sopratutto su
# una singolo gruppo di topics simili tra loro
videos_train, videos_mining_partial = train_test_split(
    multi_video_df,
    test_size=0.3,
    random_state=42,
    stratify=multi_video_df["channelId"]
)

# aggiungi i canali singoli al validation set
videos_mining = pd.concat([videos_mining_partial, single_video_df], ignore_index=True)

print(f"\nFine tuning uses: {len(videos_train)}, Mining uses: {len(videos_mining)}")

The total youtubers in the collection are: 1234
The youtuber with only one video: 47

Fine tuning uses: 51724, Mining uses: 22215


In [ ]:
from sentence_transformers import InputExample
import random

def build_training_examples(videos_df, max_pairs_per_channel=1000):
    examples = []
    grouped = videos_df.groupby("channelId")

    for channel, group in grouped:
        vids = group.sample(min(len(group), 50), random_state=42)  # limita per canale
        for i in range(len(vids)):
            for j in range(i+1, len(vids)):
                sim = semantic_similarity(vids.iloc[i]["topics"], vids.iloc[j]["topics"])
                examples.append(InputExample(
                    texts=[vids.iloc[i]["content"], vids.iloc[j]["content"]],
                    label=sim
                ))

    # aggiungiamo coppie "negative" tra canali diversi
    channels = list(grouped.groups.keys())
    for _ in range(5000):  # coppie random tra canali diversi
        ch1, ch2 = random.sample(channels, 2)
        v1 = grouped.get_group(ch1).sample(1).iloc[0]
        v2 = grouped.get_group(ch2).sample(1).iloc[0]
        sim = semantic_similarity(v1["topics"], v2["topics"])
        examples.append(InputExample(texts=[v1["content"], v2["content"]], label=sim))
    return examples

train_examples = build_training_examples(videos_train)


In [ ]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

training_model = SentenceTransformer('all-MiniLM-L6-v2')

train_dataloader = DataLoader(train_examples, batch_size=32, shuffle=True)
train_loss = losses.CosineSimilarityLoss(training_model)

training_model.fit(
          train_objectives=[(train_dataloader, train_loss)],
          epochs=2,
          warmup_steps=100
)

#[3] Data mining
Here will be putted functions for information discovery of the main pipeline to obtain the final model


In [ ]:
@profile
def hopkins(data_frame: Union[np.ndarray, pd.DataFrame], sampling_size: int) -> float:
    """Assess the clusterability of a dataset. A score between 0 and 1, a score around 0.5 express
    no clusterability and a score tending to 0 express a high cluster tendency.

    Examples
    --------
    >>> from sklearn import datasets
    >>> from pyclustertend import hopkins
    >>> X = datasets.load_iris().data
    >>> hopkins(X,150)
    0.16
    """

    if type(data_frame) == np.ndarray:
        data_frame = pd.DataFrame(data_frame)

    data_frame_sample = sample_observation_from_dataset(data_frame, sampling_size)

    sample_distances_to_nearest_neighbours = get_distance_sample_to_nearest_neighbours(
        data_frame, data_frame_sample
    )

    uniformly_selected_observations_df = simulate_df_with_same_variation(
        data_frame, sampling_size
    )

    df_distances_to_nearest_neighbours = get_nearest_sample(
        data_frame, uniformly_selected_observations_df
    )

    x = sum(sample_distances_to_nearest_neighbours)
    y = sum(df_distances_to_nearest_neighbours)

    if x + y == 0:
        raise Exception("The denominator of the hopkins statistics is null")

    return y / (x + y)[0]


def get_nearest_sample(df: pd.DataFrame, uniformly_selected_observations: pd.DataFrame):
    tree = BallTree(df, leaf_size=2)
    dist, _ = tree.query(uniformly_selected_observations, k=1)
    uniformly_df_distances_to_nearest_neighbours = dist
    return uniformly_df_distances_to_nearest_neighbours


def simulate_df_with_same_variation(
    df: pd.DataFrame, sampling_size: int
) -> pd.DataFrame:
    max_data_frame = df.max()
    min_data_frame = df.min()
    uniformly_selected_values_0 = np.random.uniform(
        min_data_frame.iloc[0], max_data_frame.iloc[0], sampling_size
    )
    uniformly_selected_values_1 = np.random.uniform(
        min_data_frame.iloc[1], max_data_frame.iloc[1], sampling_size
    )
    uniformly_selected_observations = np.column_stack(
        (uniformly_selected_values_0, uniformly_selected_values_1)
    )
    if len(max_data_frame) >= 2:
        for i in range(2, len(max_data_frame)):
            uniformly_selected_values_i = np.random.uniform(
                min_data_frame.iloc[i], max_data_frame.iloc[i], sampling_size
            )
            to_stack = (uniformly_selected_observations, uniformly_selected_values_i)
            uniformly_selected_observations = np.column_stack(to_stack)
    uniformly_selected_observations_df = pd.DataFrame(uniformly_selected_observations)
    return uniformly_selected_observations_df

def get_distance_sample_to_nearest_neighbours(df: pd.DataFrame, data_frame_sample):
    tree = BallTree(df, leaf_size=2)
    dist, _ = tree.query(data_frame_sample, k=2)
    data_frame_sample_distances_to_nearest_neighbours = dist[:, 1]
    return data_frame_sample_distances_to_nearest_neighbours


def sample_observation_from_dataset(df, sampling_size: int):
    if sampling_size > df.shape[0]:
        raise Exception("The number of sample of sample is bigger than the shape of D")
    data_frame_sample = df.sample(n=sampling_size)
    return data_frame_sample


def silhoutte(X,model):
    silhouette_list = []
    inertia_list = []
    f, axes = plt.subplots(2, 4, figsize=(20, 10))
    for n_clusters in range(2, 10):
        #kmeans = KMeans(n_clusters=n_clusters, random_state=10, n_init='auto')
        #y_pred = kmeans.fit_predict(X)
        model_hyp=model(n_clusters=n_clusters, random_state=10, n_init='auto')
        y_pred=model_hyp.fit_predict(X)

        """
            models=[KMeans(),
            kmedoids(),
            clarans(),
            AgglomerativeClustering(),
            Birch(),
            DBSCAN(),
            OPTICS(),
            clique(),
            HDBSCAN()
            ]

        """

        # evaluate silhouette score
        silhouette_avg = silhouette_score(X, y_pred)
        silhouette_list.append(silhouette_avg)

        # evaluate inertia
        inertia = model_hyp.inertia_
        inertia_list.append(inertia)

        # display clustered samples
        axes[(n_clusters - 2) // 4][(n_clusters - 2) % 4].scatter(X.iloc[:, 0], X.iloc[:, 1], c=y_pred, alpha=0.5)
        axes[(n_clusters - 2) // 4][(n_clusters - 2) % 4].axis('equal')
        axes[(n_clusters - 2) // 4][(n_clusters - 2) % 4].set_xlabel('Feature 1')
        axes[(n_clusters - 2) // 4][(n_clusters - 2) % 4].set_ylabel('Feature 2')
        axes[(n_clusters - 2) // 4][(n_clusters - 2) % 4].set_title(
            f'k = {n_clusters} - Avg. Silhouette = {silhouette_avg:.2} - n_iter = {model_hyp.n_iter_}')

        # display clusters centroids
        centers = model_hyp.cluster_centers_
        axes[(n_clusters - 2) // 4][(n_clusters - 2) % 4].scatter(centers[:, 0], centers[:, 1], marker='x', c='r')

    plt.tight_layout()

    # plot silhouette and inertia trends w.r.t the number of clusters
    fig, ax1 = plt.subplots()
    ax1.set_xlabel('k')
    ax1.set_ylabel('avg-silhouette', color='black')
    ax1.plot(range(2, 10), silhouette_list, '--ok')
    ax1.tick_params(axis='y', labelcolor='black')
    ax1.grid(axis='y')

    ax2 = ax1.twinx()
    ax2.set_ylabel('loss', color='red')
    ax2.plot(range(2, 10), inertia_list, '--or', alpha=0.2)
    ax2.tick_params(axis='y', labelcolor='red')

    plt.tight_layout()  # otherwise the right y-label is slightly clipped

    plt.show()

def centroidImpact(X,n=4):
    avg_sil_random = [silhouette_score(X, KMeans(n_clusters=n, init='random', random_state=i, n_init=1).fit_predict(X)) for i in range(100)]

    avg_sil_plusplus = [silhouette_score(X,KMeans(n_clusters=n, init='k-means++', random_state=i, n_init=1).fit_predict(X)) for i in range(100)]

    df = pd.DataFrame({'random': avg_sil_random, 'km++': avg_sil_plusplus})
    df.boxplot(ylabel='Avg. Silhouette')
    plt.show()

    axes = df.hist(sharey=True)
    for ax in axes.flatten():
        ax.set_xlabel('Avg. Silhouette')


def qualityMetrics(X,y_groundTruth,y,n=4):
    print(f'kmeans with n_cluster = {n} --> ARI = {ARI(y, y_groundTruth): .03}')
    CM(y, y_groundTruth)
    homogeneity, completeness, v_measure = HCV(y, y_groundTruth)
    print(f'homogeneity: {homogeneity: .3}')
    print(f'completeness: {completeness: .3}')
    print(f'v_measure: {v_measure: .3}')

In [ ]:

# Load a pre-trained SentenceTransformer model
colList = [f"c{i}" for i in range(1, 1025)]
@profile
def sbertCalc(fileName="videos_documents.csv"):
    print("SBERT step")
    model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")
    videosDf=pd.read_csv(os.path.join(path_prefix,"Results",fileName))
    videosDf=videosDf.head(500)
    videosSbert=videosDf.copy()
    embeddings = model.encode_document(list(videosDf['content']))
    print(embeddings.shape)
    data=pd.DataFrame(embeddings,columns=colList)
    videosSbert=pd.concat([videosSbert,data],axis=1)
    return videosSbert

sbertDF=sbertCalc()
sbertDF.to_csv(os.path.join(path_prefix,"Results","videos_sbert.csv"),index=False)

In [ ]:
@profile
def clustering(fileName="videos_sbert.csv"):
    df=pd.read_csv(os.path.join(path_prefix,"Results",fileName))
    print("CLUSTERING step (Hopkins)")
    X=df.iloc[:,4:]

    #Compute Hopkins statistic
    print(f"Hopkins stat: ",hopkins(X,df.shape[0]))

    #Little visualization
    plt.figure(figsize=(7, 7))
    plt.scatter(X.iloc[:, 0], X.iloc[:, 1])
    plt.axis('equal') # force equal axes aspect ratio
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

    #model_hyp=model(n_clusters=n_clusters, random_state=10, n_init='auto')
    #y_pred=model_hyp.fit_predict(X)
    kmeans = KMeans(n_clusters=2, random_state=10, n_init='auto')
    y_pred = kmeans.fit_predict(X)
    print(y_pred)

    model=kmedoids(X.values,[1,2])
    model.process();
    clusters = model.get_clusters()
    n_samples = max(max(c) for c in clusters) + 1
    y_pred = np.empty(n_samples, dtype=int)
    for cluster_id, indices in enumerate(clusters):
        y_pred[indices] = cluster_id
    print(y_pred)



    plt.figure(figsize=(7, 7))
    plt.scatter(X.iloc[:, 0], X.iloc[:, 1],c=y_pred)
    plt.axis('equal') # force equal axes aspect ratio
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

    """
    models=[KMeans] # Pass the class, not an instance

    for model in models:
        silhoutte(X,model)

    models=[KMeans(),
            kmedoids(),
            clarans(),
            AgglomerativeClustering(),
            Birch(),
            DBSCAN(),
            OPTICS(),
            clique(),
            HDBSCAN()
            ]

    #finding number of clustering k
    silhoutte(X)
    n=4

    #clustering
    kmeans = KMeans(n_clusters=n, random_state=10, n_init='auto')
    y = kmeans.fit_predict(X)

    #Quality of clustering
    centroidImpact(X,n)
        #with ground truth
    #qualityMetrics(X,y_groundTruth,y,n)
    """

clustering()